In [ ]:
import os
import numpy as np
import pandas as pd


class CustomerIntelligence:

    def __init__(self, filepath):

        self.filepath = filepath

        os.makedirs("../outputs/customer", exist_ok=True)

    ###########################################################

    def load_data(self):

        print("=" * 60)
        print("Loading Customer Dataset")
        print("=" * 60)

        self.df = pd.read_csv(
            self.filepath,
            parse_dates=["InvoiceDate"],
            low_memory=False,
            dtype={
                "Invoice": str,
                "StockCode": str
            }
        )

        print("Dataset Shape:", self.df.shape)

    ###########################################################

    def customer_statistics(self):

        print("\nCreating Customer Statistics...\n")

        latest = self.df["InvoiceDate"].max()

        self.customer = (

            self.df

            .groupby("Customer_ID")

            .agg(

                Total_Revenue=("Total_Price", "sum"),

                Total_Orders=("Invoice", "nunique"),

                Total_Quantity=("Quantity", "sum"),

                Avg_Order_Value=("Total_Price", "mean"),

                First_Purchase=("InvoiceDate", "min"),

                Last_Purchase=("InvoiceDate", "max"),

                Countries=("Country", "nunique"),

                Products=("StockCode", "nunique")

            )

            .reset_index()

        )

        self.customer["Customer_Age_Days"] = (

            latest -

            self.customer["First_Purchase"]

        ).dt.days

        self.customer["Recency"] = (

            latest -

            self.customer["Last_Purchase"]

        ).dt.days

        print("Customer Dataset Shape:", self.customer.shape)

    ###########################################################

    def customer_lifetime_value(self):

        print("\nCalculating Customer Lifetime Value...\n")

        self.customer["Purchase_Frequency"] = (

            self.customer["Total_Orders"]

            /

            self.customer["Customer_Age_Days"]

            .replace(0, 1)

        )

        self.customer["CLV"] = (

            self.customer["Avg_Order_Value"]

            *

            self.customer["Purchase_Frequency"]

            *

            365

        ).round(2)

    ###########################################################

    def customer_tier(self):

        print("\nAssigning Customer Tier...\n")

        q1 = self.customer["CLV"].quantile(0.25)
        q2 = self.customer["CLV"].quantile(0.50)
        q3 = self.customer["CLV"].quantile(0.75)

        def tier(x):

            if x >= q3:
                return "Platinum"

            elif x >= q2:
                return "Gold"

            elif x >= q1:
                return "Silver"

            else:
                return "Bronze"

        self.customer["Tier"] = (

            self.customer["CLV"]

            .apply(tier)

        )

        print(self.customer["Tier"].value_counts())

    ###########################################################

    def churn_risk(self):

        print("\nCalculating Churn Risk...\n")

        r75 = self.customer["Recency"].quantile(0.75)

        f25 = self.customer["Total_Orders"].quantile(0.25)

        def risk(row):

            if (

                row["Recency"] > r75

                and

                row["Total_Orders"] <= f25

            ):

                return "High"

            elif row["Recency"] > r75:

                return "Medium"

            else:

                return "Low"

        self.customer["Churn_Risk"] = (

            self.customer

            .apply(risk, axis=1)

        )

        print(self.customer["Churn_Risk"].value_counts())

    ###########################################################

    def customer_health(self):

        print("\nCreating Customer Health Score...\n")

        def health(row):

            if (

                row["Tier"] == "Platinum"

                and

                row["Churn_Risk"] == "Low"

            ):

                return "Excellent"

            elif row["Tier"] in ["Gold", "Platinum"]:

                return "Good"

            elif row["Tier"] == "Silver":

                return "Average"

            else:

                return "Poor"

        self.customer["Health"] = (

            self.customer

            .apply(health, axis=1)

        )

        print(self.customer["Health"].value_counts())

    ###########################################################

    def save(self):

        columns = [

            "Customer_ID",

            "Total_Revenue",

            "Total_Orders",

            "Total_Quantity",

            "Avg_Order_Value",

            "First_Purchase",

            "Last_Purchase",

            "Customer_Age_Days",

            "Recency",

            "Countries",

            "Products",

            "Purchase_Frequency",

            "CLV",

            "Tier",

            "Churn_Risk",

            "Health"

        ]

        self.customer = self.customer[columns]

        self.customer.to_csv(

            "../outputs/customer/customer_360.csv",

            index=False

        )

        print("\nCustomer 360 Dataset Saved Successfully")

    ###########################################################

    def run(self):

        self.load_data()

        self.customer_statistics()

        self.customer_lifetime_value()

        self.customer_tier()

        self.churn_risk()

        self.customer_health()

        self.save()

        print("\nCustomer Intelligence Completed")


###########################################################

if __name__ == "__main__":

    obj = CustomerIntelligence(

        r"C:\Users\ka843\Coding\Amdox Internship_project\data\clean\customer_sales.csv"

    )

    obj.run()